In [1]:
import boto3

profile_name = "odin-cdk"
session = boto3.Session(profile_name=profile_name)
credentials = session.get_credentials()

In [2]:
from typing import List, TypedDict
import boto3
import json
from datetime import datetime
from os import environ
import pandas as pd

STATE_MACHINE = (
    "arn:aws:states:eu-north-1:991049544436:stateMachine:OdinSMROdincalStateMachine"
)


class State(TypedDict):
    job: str
    status: str
    date: datetime | None
    error: str | None


sfn_client = boto3.client(
    "stepfunctions",
    region_name="eu-north-1",
    aws_access_key_id=credentials.access_key,
    aws_secret_access_key=credentials.secret_key,
)
History = List[State]

In [3]:
def get_history(nmax: int) -> History:
    history: History = []
    paginator = sfn_client.get_paginator("list_executions")
    response_iterator = paginator.paginate(
        stateMachineArn=STATE_MACHINE,
        statusFilter="FAILED",
        PaginationConfig={
            "MaxItems": nmax,
            "PageSize": 100,
        },
    )
    for response in response_iterator:
        for exec in response["executions"]:
            info = sfn_client.describe_execution(executionArn=exec["executionArn"])
            input = json.loads(info["input"])
            if "name" in input:
                history.append(
                    State(
                        job=input["name"],
                        status=exec["status"],
                        date=exec.get("stopDate", None),
                        error=info.get("error"),
                    )
                )
    return history

In [4]:
def run_last_failed(hist: History) -> None:
    df = pd.DataFrame.from_dict(hist)
    df.sort_values(by=["job", "date"]).drop_duplicates(subset="job", keep="last")
    jobs = df[df.error == "NoERA5DataError"]["job"].to_list()
    for j in jobs:
        type = "ac2"
        if j.startswith("ac1"):
            type = "ac1"
        sfn_input = json.dumps(dict(name=j, type=type))
        sfn_client.start_execution(stateMachineArn=STATE_MACHINE, input=sfn_input)

In [ ]:
hist = get_history(5)
# run_last_failed(hist)